##### Imports

In [44]:
# %pip install -r requirements.txt

In [45]:
import pandas as pd
import numpy as np
import plotly.express as px
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
from typing import Dict, Tuple
from scipy.stats import norm, invgamma, multivariate_normal
import pickle
import os

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

warnings.filterwarnings('ignore')


from kama_msr import KAMA
from kama_msr import MarkovSwitchingModel
from kama_msr import KAMA_MSR

# Getting Price Data

#### -------------------------------------------------------------------------------------------------

In [46]:
# Prepare data
international_index_symbol_names = pd.read_csv('data/inputs/fmp_index_list.csv').set_index('symbol')['name']
international_index_symbol_names = international_index_symbol_names[~international_index_symbol_names.index.isin(['^GSPC', '^NDX'])].to_dict()
commodity_symbol_names = pd.read_csv('data/inputs/fmp_commodity_list.csv').set_index('symbol')['name'].to_dict()

etf_symbol_names = {
    # BOND ETFS
    # 'BIL': 'SPDR Bloomberg 1-3 Month T-Bill ETF',
    # 'SHY': 'iShares 1-3 Year Treasury Bond ETF',
    # 'IEF': 'iShares 7-10 Year Treasury Bond ETF',
    
    # MAJOR INDICES
    '^GSPC': 'S&P 500',
    '^IXIC': 'Nasdaq Composite',
    '^NDX': 'Nasdaq 100',
    '^RUT': 'Russell 2000',
    '^DJI': 'Dow Jones Industrial Average',
    
    # Commodity ETFs
    'GLD': 'SPDR Gold Shares',
    'SLV': 'iShares Silver Trust',
    'USO': 'United States Oil Fund',
    'UNG': 'United States Natural Gas Fund',
    'DBA': 'Invesco DB Agriculture',
    'CPER': 'United States Copper Index Fund',
    'DBB': 'Invesco DB Base Metals',
    'PALL': 'abrdn Palladium Shares ETF',
    'PLTM': 'GraniteShares Platinum Trust',
    'WEAT': 'Teucrium Wheat Fund',
    'SOYB': 'Teucrium Soybean Fund',
    'CORN': 'Teucrium Corn Fund',
    'CANE': 'Teucrium Sugar Fund',
    
    # MAIN BROAD MARKET ETFS
    'SPY': 'SPDR S&P 500 ETF',
    # 'VOO': 'Vanguard S&P 500 ETF',
    # 'RSP': 'Invesco S&P 500 Equal Weight ETF',
    # 'IVV': 'iShares Core S&P 500 ETF',
    'QQQ': 'Invesco QQQ Trust',
    'IWM': 'iShares Russell 2000 ETF',
    # 'IWB': 'iShares Russell 1000 ETF',
    'DIA': 'SPDR Dow Jones Industrial Average ETF',
    # 'VTI': 'Vanguard Total Stock Market ETF',
    
    # S&P 500 SECTOR ETFS
    'XLE': 'Energy Select Sector SPDR',
    'XLF': 'Financial Select Sector SPDR',
    'XLU': 'Utilities Select Sector SPDR',
    'XLI': 'Industrial Select Sector SPDR',
    'XLV': 'Health Care Select Sector SPDR',
    'XLK': 'Technology Select Sector SPDR',
    'XLB': 'Materials Select Sector SPDR',
    'XLY': 'Consumer Discretionary Select Sector SPDR',
    'XLP': 'Consumer Staples Select Sector SPDR',
    # 'XLRE': 'Real Estate Select Sector SPDR',
    # 'XLC': 'Communication Services Select Sector SPDR',
    'IYR': 'iShares U.S. Real Estate ETF',
    'IYZ': 'iShares U.S. Telecommunications ETF',
    
    # GROWTH ETFs
    'MGK': 'Vanguard Mega Cap Growth ETF',
    'IVW': 'iShares S&P 500 Growth ETF',
    # 'IWF': 'iShares Russell 1000 Growth ETF',
    'IWO': 'iShares Russell 2000 Growth ETF',
    # 'VUG': 'Vanguard Growth ETF',
    
    # VALUE ETFs
    'MGV': 'Vanguard Mega Cap Value ETF',
    'IVE': 'iShares S&P 500 Value ETF',
    # 'IWD': 'iShares Russell 1000 Value ETF',
    'IWN': 'iShares Russell 2000 Value ETF',
    # 'VTV': 'Vanguard Value ETF',
    
    # SIZE ETFs
    'OEF': 'iShares S&P 100 ETF',
    'IWR': 'iShares Russell Mid-Cap ETF',
    'IWC': 'iShares Micro-Cap ETF',
    # 'IJH': 'iShares Core S&P Mid-Cap ETF',
    # 'IJR': 'iShares Core S&P Small-Cap ETF',
    # 'MDY': 'SPDR S&P MidCap 400 ETF',
    
    # INTERNATIONAL
    # 'VXUS': 'Vanguard Total International Stock ETF',
    'VEA': 'Vanguard FTSE Developed Markets ETF',
    'VWO': 'Vanguard FTSE Emerging Markets ETF',
    'VGK': 'Vanguard FTSE Europe ETF',
    'VPL': 'Vanguard FTSE Pacific ETF',
    'FXI': 'iShares China Large-Cap ETF',
    'EWJ': 'iShares MSCI Japan ETF',
    'INDA': 'iShares MSCI India ETF',
    # 'EFA': 'iShares MSCI EAFE ETF',
    'EEM': 'iShares MSCI Emerging Markets ETF',
}

universe_symbol_names = {
    'IVV': 'IVV - iShares Core S&P 500 ETF',
    'IJH': 'IJH - iShares Core S&P Mid-Cap ETF',
    'IWM': 'IWM - iShares Russell 2000 ETF',
    'EFA': 'EFA - iShares MSCI EAFE ETF',
    'EEM': 'EEM - iShares MSCI Emerging Markets ETF',
    'AGG': 'AGG - iShares Core U.S. Aggregate Bond ETF',
    'SPTL': 'SPTL - SPDR Portfolio Long Term Treasury ETF',
    'HYG': 'HYG - iShares iBoxx $ High Yield Corporate Bond ETF',
    'SPBO': 'SPBO - SPDR Portfolio Corporate Bond ETF',
    'IYR': 'IYR - iShares U.S. Real Estate ETF',
    'DBC': 'DBC - Invesco DB Commodity Index Tracking Fund',
    'GLD': 'GLD - SPDR Gold Shares',
}

# international_index_data = pd.read_csv('data/processed/index_data.csv', index_col=0, header=[0, 1], parse_dates=True)
# commodity_data = pd.read_csv('data/processed/commodity_data.csv', index_col=0, header=[0, 1], parse_dates=True)
etf_data = pd.read_csv('data/processed/all_etf_data.csv', index_col=0, header=[0, 1], parse_dates=True)
# universe_data = pd.read_csv('data/processed/universe_etfs.csv', index_col=0, header=[0, 1], parse_dates=True)

# commodity_data_close_cols = commodity_data.columns[commodity_data.columns.get_level_values(1) == 'close']
# commodity_close_prices = commodity_data[commodity_data_close_cols].droplevel(1, axis=1).rename(columns=commodity_symbol_names)
# commodity_close_prices.columns = [col.replace('/', ' ') for col in commodity_close_prices.columns]

etf_close_cols = etf_data.columns[etf_data.columns.get_level_values(1) == 'close']
etf_close_prices = etf_data[etf_close_cols].droplevel(1, axis=1).rename(columns=etf_symbol_names)

# universe_close_cols = universe_data.columns[universe_data.columns.get_level_values(1) == 'close']
# universe_close_prices = universe_data[universe_close_cols].droplevel(1, axis=1).rename(columns=universe_symbol_names)

In [47]:
etf_always_disclude = ['Vanguard S&P 500 ETF', 'Real Estate Select Sector SPDR', 'Communication Services Select Sector SPDR']
etf_disclude = [name for name in etf_close_prices.columns if 'Russell 1000' in name or 'Russell 3000' in name]\
                        + ['S&P 500', 'Nasdaq Composite', 'Dow Jones Industrial Average', 'Nasdaq 100', 'Russell 2000']

etf_include = list(set(etf_close_prices.columns.tolist()) - set(etf_always_disclude) - set(etf_disclude))
etf_close_prices = etf_close_prices[etf_include]

# us_treasury = ['SPDR Bloomberg 1-3 Month T-Bill ETF', 'iShares 1-3 Year Treasury Bond ETF', 'iShares 7-10 Year Treasury Bond ETF']
# int_equity = ['Vanguard Total International Stock ETF', 'Vanguard FTSE Developed Markets ETF',\
#                                         'Vanguard FTSE Emerging Markets ETF','Vanguard FTSE Europe ETF',\
#                                         'Vanguard FTSE Pacific ETF', 'iShares China Large-Cap ETF',\
#                                         'iShares MSCI Japan ETF', 'iShares MSCI India ETF']
us_equity = list(set(etf_include))

#### -------------------------------------------------------------------------------------------------

In [55]:
df = etf_close_prices['SPDR S&P 500 ETF'].to_frame().dropna()
rebal_dates = df.loc['2018-12-31':].index[::21]

rebal_dates_to_use = list(reversed(rebal_dates))
print(len(rebal_dates_to_use))

82


# Fitting KAMA+MSR

In [56]:
def fit_KAMA_MSR(sdte: datetime | str, 
                 edte: datetime | str,
                 asset_names: list[str],
                 n_regimes: int,
                 use_three_state_msr: bool,
                 kama_params: Dict,
                 filter_params: Dict,
                 n_samples: int,
                 burnin: int,
                 thin: int,
                 close_prices: pd.DataFrame,
                 # NEW PARAMETERS FOR IMPROVEMENTS
                 optimize_kama: bool = True,
                 kama_optimization_method: str = 'random',
                 n_random_trials: int = 50,
                 min_regime_duration: int | None = None,
                 duration_enforcement_method: str = 'extend',
                 random_seed: int | None = None,
                 msr_verbose: bool = True) -> Dict[str, 'KAMA_MSR']:
    """
    Fit KAMA+MSR models with improved optimization and duration enforcement.
    
    Parameters:
    -----------
    sdte : datetime | str
        Start date for data
    edte : datetime | str
        End date for data
    asset_names : list[str]
        List of asset names to process
    n_regimes : int
        Number of MSR regimes (2 or 3)
    use_three_state_msr : bool
        Whether to use 3-state MSR
    kama_params : dict
        Initial KAMA parameters (will be overridden if optimize_kama=True)
    filter_params : dict
        Filter parameters
    n_samples : int
        Number of MCMC samples
    burnin : int
        MCMC burnin period
    thin : int
        MCMC thinning
    close_prices : pd.DataFrame
        DataFrame of close prices
    
    NEW PARAMETERS:
    ---------------
    optimize_kama : bool, default=False
        Whether to optimize KAMA parameters using misclassification score
    kama_optimization_method : str, default='random'
        Optimization method: 'random' (fast) or 'coarse_to_fine' (thorough)
    n_random_trials : int, default=50
        Number of random trials for optimization (if method='random')
    optimize_filter : bool, default=False
        Whether to optimize filter parameters
    min_regime_duration : int | None, default=None
        Minimum number of periods a regime must persist
        If None, no duration enforcement
    duration_enforcement_method : str, default='extend'
        Method for duration enforcement: 'extend', 'merge', or 'majority'
    random_seed : int | None, default=None
        Random seed for reproducibility
    msr_verbose : bool, default=True
        Whether to print MSR fitting progress
    
    Returns:
    --------
    Dict[str, KAMA_MSR] : Dictionary of fitted models by asset name
    """
    models = {}
    
    for asset_name in asset_names:
        if asset_name not in close_prices.columns:
            print(f"Asset name {asset_name} not found in provided close prices data. Skipping.")
            continue
        
        print(f"\n{'='*160}")
        print(f"PROCESSING: {asset_name}")
        print(f"{'='*160}")

        # Prepare data
        prices = close_prices[asset_name].dropna()
        prices = prices.loc[sdte:edte]
        
        # Initialize model
        model = KAMA_MSR(
            kama_params=kama_params,
            msr_params={'n_regimes': n_regimes},
            filter_params=filter_params,
            use_three_state_msr=use_three_state_msr,
            random_seed=random_seed  # NEW: Set random seed
        )
        
        # Fit the model with improvements
        model.fit(
            asset_name=asset_name,
            prices=prices,
            
            # NEW: KAMA optimization using misclassification score
            optimize_kama=optimize_kama,
            kama_optimization_method=kama_optimization_method,
            n_random_trials=n_random_trials,
            
            # # Filter optimization (optional)
            # optimize_filter=optimize_filter,
            
            # NEW: Minimum regime duration enforcement
            min_regime_duration=min_regime_duration,
            duration_enforcement_method=duration_enforcement_method,
            
            # MSR parameters
            msr_verbose=msr_verbose,
            n_samples=n_samples,
            burnin=burnin,
            thin=thin
        )
        
        # Print summary statistics
        print(f"\n{'='*80}")
        print(f"SUMMARY FOR {asset_name}")
        print(f"{'='*80}")
        
        if optimize_kama:
            print(f"Optimized KAMA: n={model.kama.n}, n_fast={model.kama.n_fast}, "
                  f"n_slow={model.kama.n_slow}, gamma={model.gamma:.3f}")
        
        if min_regime_duration is not None:
            print(f"\nDuration Statistics:")
            print(model.analyze_regime_durations().to_string(index=False))
        
        print(f"\nRegime Distribution:")
        regime_counts = model.regime_labels.value_counts().sort_index()
        total = len(model.regime_labels)
        for regime, count in regime_counts.items():
            print(f"  Regime {regime}: {count:5d} periods ({100*count/total:5.1f}%)")
        
        n_changes = (model.regime_labels.diff() != 0).sum()
        avg_duration = total / (n_changes + 1)
        print(f"\nRegime Changes: {n_changes}")
        print(f"Average Duration: {avg_duration:.1f} periods")
        
        models[asset_name] = model
        
    print(f"\n{'='*80}")
    print(f"COMPLETED FITTING {len(models)} ASSETS")
    print(f"{'='*80}\n")
    
    return models

def save_KAMA_MSR_models(models: Dict[str, 'KAMA_MSR'], 
                         asset_type_sub_folder: str, 
                         edte_sub_folder: str,
                         save_metadata: bool = True) -> None:
    """
    Save fitted KAMA+MSR models with metadata.
    
    Parameters:
    -----------
    models : Dict[str, KAMA_MSR]
        Dictionary of fitted models
    asset_type_sub_folder : str
        Sub-folder for asset type (e.g., 'us_equity')
    edte_sub_folder : str
        Sub-folder for end date (e.g., '20230101')
    save_metadata : bool, default=True
        Whether to save metadata file with model info
    """
    save_dir = f'saved_models/KAMA_MSR/{asset_type_sub_folder}/{edte_sub_folder}'
    os.makedirs(save_dir, exist_ok=True)
    
    metadata = {}
    
    for asset_name, model in models.items():
        # Clean asset name for filename
        clean_name = asset_name.replace('/', '_').replace('\\', '_')
        n_regimes = model.msr.n_regimes
        total_regimes = n_regimes * 2
        
        # Save model
        filename = f'{clean_name}_KAMA-MSR_{total_regimes}-regimes.pkl'
        filepath = os.path.join(save_dir, filename)
        
        with open(filepath, 'wb') as f:
            pickle.dump(model, f)
        
        print(f"✓ Saved: {filepath}")
        
        # Collect metadata
        if save_metadata:
            metadata[asset_name] = {
                'filename': filename,
                'n_msr_regimes': n_regimes,
                'n_combined_regimes': total_regimes,
                'kama_params': {
                    'n': model.kama.n,
                    'n_fast': model.kama.n_fast,
                    'n_slow': model.kama.n_slow
                },
                'filter_params': {
                    'n_lookback': model.n_lookback,
                    'gamma': model.gamma
                },
                'min_regime_duration': model.min_regime_duration,
                'duration_method': model.duration_method,
                'n_data_points': len(model.regime_labels),
                'regime_distribution': model.regime_labels.value_counts().to_dict(),
                'n_regime_changes': (model.regime_labels.diff() != 0).sum()
            }
    
    # Save metadata
    if save_metadata and metadata:
        metadata_file = os.path.join(save_dir, 'metadata.pkl')
        with open(metadata_file, 'wb') as f:
            pickle.dump(metadata, f)
        print(f"\n✓ Saved metadata: {metadata_file}")

from joblib import Parallel, delayed
import multiprocessing
print("Total CPUs:", multiprocessing.cpu_count())
# use_cpus = int(multiprocessing.cpu_count() / 2)
use_cpus = 3
print("Using CPUs:", use_cpus)

Total CPUs: 8
Using CPUs: 3


In [57]:
print(len(us_equity))

from pathlib import Path
asset_names_fitted = list(Path('saved_models/KAMA_MSR/us_equity/20181231').glob('*_KAMA-MSR_4-regimes.pkl'))
asset_names_fitted = [file.stem.split('_')[0] for file in asset_names_fitted]
print(len(asset_names_fitted))
set(us_equity) - set(asset_names_fitted)

43
31


{'GraniteShares Platinum Trust',
 'Invesco DB Base Metals',
 'Teucrium Corn Fund',
 'Teucrium Soybean Fund',
 'Teucrium Sugar Fund',
 'Teucrium Wheat Fund',
 'United States Copper Index Fund',
 'Vanguard Mega Cap Growth ETF',
 'Vanguard Mega Cap Value ETF',
 'abrdn Palladium Shares ETF',
 'iShares S&P 100 ETF',
 'iShares U.S. Telecommunications ETF'}

In [58]:
asset_names_to_fit = set(us_equity) - set(asset_names_fitted)
print(len(asset_names_to_fit))
asset_names_to_fit

12


{'GraniteShares Platinum Trust',
 'Invesco DB Base Metals',
 'Teucrium Corn Fund',
 'Teucrium Soybean Fund',
 'Teucrium Sugar Fund',
 'Teucrium Wheat Fund',
 'United States Copper Index Fund',
 'Vanguard Mega Cap Growth ETF',
 'Vanguard Mega Cap Value ETF',
 'abrdn Palladium Shares ETF',
 'iShares S&P 100 ETF',
 'iShares U.S. Telecommunications ETF'}

In [ ]:
def fit_one_asset(asset_name, rebal_date):
    # Inputs
    sdte = datetime(1995, 1, 1)
    edte = rebal_date
    asset_names = [asset_name] # us_equity, us_treasury, int_equity, commodity_close_prices.columns.tolist(), international_index_close_prices.columns.tolist()
    close_prices = etf_close_prices # etf_close_prices, commodity_close_prices, international_index_close_prices
    asset_type_sub_folder = 'us_equity' # 'us_equity', 'us_treasury', 'int_equity', 'commodity'
    n_regimes=2
    use_three_state_msr = (n_regimes == 3)
    kama_params = {'n': 20, 'n_fast': 5, 'n_slow': 30}
    filter_params = {'n_lookback': False, 'gamma': 1}
    n_samples=1000
    burnin=200
    thin=1
    min_regime_duration = 1

    models = fit_KAMA_MSR(
        sdte=sdte,
        edte=edte,
        asset_names=asset_names,
        n_regimes=n_regimes,
        use_three_state_msr=False,
        kama_params=kama_params,
        filter_params=filter_params,
        n_samples=n_samples,
        burnin=burnin,
        thin=thin,
        close_prices=close_prices,
        # Full optimization
        optimize_kama=True,
        kama_optimization_method='coarse_to_fine',              
        min_regime_duration=min_regime_duration,
        duration_enforcement_method='merge',
        random_seed=1010,
        msr_verbose=True
    )

    save_KAMA_MSR_models(models, asset_type_sub_folder, edte.strftime('%Y%m%d'), save_metadata=False)

for i, rebal_date in enumerate(rebal_dates_to_use):
    print(f'Fitting for rebalance date {i+1}/{len(rebal_dates_to_use)}:', rebal_date)
    results = Parallel(n_jobs=use_cpus)(
        delayed(fit_one_asset)(asset, rebal_date) for asset in asset_names_to_fit
    )

Fitting for rebalance date 1/82: 2025-10-07 00:00:00



================================================================================================================================================================PROCESSING: United States Copper Index FundPROCESSING: Teucrium Soybean Fund


================================================================================================================================================================PROCESSING: iShares S&P 100 ETF================================================================================================================================================================




KAMA+MSR COMBINED MODEL FITTING for Teucrium Soybean Fund

Mode: 4-Regime (2-State MSR)

[1/5] Fitting 2-state MSR model...

KAMA+MSR COMBINED MODEL FITTING for United States Copper Index Fund
Mode: 4-Regime (2-State MSR)

[1/5] Fitting 2-state MSR model...
KAMA+MSR COMBINED MODEL FITTING for iShares S&P 100 ETF
Mode: 4-Regime (2-State MSR)

[1/5]